# DUT power-trace capture (CW305 + VCU118 Ethernet)

The DUT runs on the **CW305**. The **VCU118 + AD9467** captures a 1024-sample window on each DUT `busy` rising edge (trigger position 80), queues traces on-chip, and UDP-broadcasts them (`0xAE511ACE`) to the host at high rate.

**Clocks:** PLL out1 = 20 MHz (DUT), out2 = 100 MHz (CLKOUT SMA → AD9467).

**Wires (CW305 → VCU118 J52 PMOD0):**
| CW305 | VCU J52 | Signal |
|---|---|---|
| A15 | pin 1 (AY14) | load ← |
| E15 | pin 3 (AY15) | busy → |
| B14 | pin 5 (AW15) | ready → |
| D16 | pin 2 (AV16) | done → (session end) |
| GND | pin 9 | ground |

Do **not** connect 3.3 V on pins 11/12.

**Ethernet:** PC USB Ethernet ↔ VCU118 J10. Bind RX to that NIC only.

**Bring-up order:** USB-reset + force-program CW305 → enable PLL 20/100 MHz → wait for AD9467 DCO → program VCU bit + `.ltx` in Vivado → start eth RX → run captures.

**Bitstreams (from this repo):** `../bitstreams/cw305_top.bit`, `../bitstreams/vcu118_capture_top.bit`.


In [ ]:
import chipwhisperer as cw
from Crypto.Cipher import AES
import matplotlib.pyplot as plt
import numpy as np
import socket
import struct
import threading
import time
import os

print("Devices:")
for d in cw.list_devices():
    print(d)

SCOPE_SN  = "50203220343043543130353139313037"
TARGET_SN = "5020322030384a583030313230313039"
FPGA_ID   = "100t"
BITFILE   = r"../bitstreams/cw305_top.bit"
# Do not pass custom defines_files with REG_HS_RST (CW305.py expects 12 defines).
# Use REG_HS_RST = 0x0C as a raw fpga_write address instead.

# Required: USB-reset then force-reprogram so the bitfile actually loads.
RESET_BEFORE_PROGRAM = True

KEY       = bytearray([0x00] * 16)
PLAINTEXT = bytearray([0x00, 0x11, 0x22, 0x33, 0x44, 0x55, 0x66, 0x77,
                       0x88, 0x99, 0xaa, 0xbb, 0xcc, 0xdd, 0xee, 0xff])

N_TRACES = 20
REG_HS_RST = 0x0C
MAGIC = 0xAE511ACE
UDP_PORT = 9
HDR_FMT = ">IIBBH"
HDR_LEN = struct.calcsize(HDR_FMT)

def usb_eth_bind_ips():
    """Prefer the USB Ethernet NIC that goes to VCU J10 (not the internet NIC)."""
    ips = []
    try:
        import psutil
        addrs = psutil.net_if_addrs()
        for name, snics in addrs.items():
            nl = name.lower()
            if not ("ethernet 2" in nl or "usb eth" in nl or "displaylink" in nl):
                continue
            for s in snics:
                if getattr(s, "family", None) == socket.AF_INET:
                    ips.append(s.address)
                    print(f"USB eth NIC {name}: {s.address}")
    except Exception as e:
        print("psutil NIC scan skipped:", e)
    prefer = [ip for ip in ips if ip.startswith("192.168.1.")]
    prefer += [ip for ip in ips if ip.startswith("169.254.")]
    prefer += ["0.0.0.0"]
    out = []
    for ip in prefer:
        if ip not in out:
            out.append(ip)
    return out

print("bind candidates:", usb_eth_bind_ips())


## Scope + program CW305 (USB reset → force)

In [ ]:
scope = cw.scope(sn=SCOPE_SN)
scope.default_setup()
scope.adc.samples = 160 if scope._is_husky else 250
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"
scope.gain.db = 25

assert os.path.isfile(BITFILE), f"Missing bitstream: {BITFILE}"

if RESET_BEFORE_PROGRAM:
    print("USB-reset CW305...")
    tmp = cw.target(
        scope, cw.targets.CW305, force=False, fpga_id=FPGA_ID,
        platform="cw305", bsfile=BITFILE, sn=TARGET_SN,
    )
    tmp._naeusb.reset()
    time.sleep(1.0)
    try:
        tmp.dis()
    except Exception:
        pass
    time.sleep(0.5)


In [ ]:
target = cw.target(
    scope, cw.targets.CW305, force=True, fpga_id=FPGA_ID,
    platform="cw305", bsfile=BITFILE, sn=TARGET_SN,
)


In [ ]:
target.vccint_set(1.0)
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(True, 2)
target.pll.pll_outfreq_set(20E6, 1)
target.pll.pll_outfreq_set(100E6, 2)
target.clkusbautooff = True
target.clksleeptime = 1

if scope._is_husky:
    scope.clock.clkgen_freq = 10e6
    scope.clock.clkgen_src = "extclk"
    scope.clock.adc_mul = 4
else:
    scope.clock.adc_src = "extclk_x4"

for _ in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break
print("adc_locked=", scope.clock.adc_locked, " PLL1=", target.pll.pll_outfreq_get(1),
      " PLL2=", target.pll.pll_outfreq_get(2))

# Clear stale handshake; asserts D16 done pulse toward VCU
target.fpga_write(REG_HS_RST, [1])
time.sleep(0.15)
print("REG_HS_RST done — wait for VCU DCO warm")
time.sleep(2.0)

## Ethernet RX on USB NIC

Program VCU118 (`vcu118_capture_top.bit` + `.ltx`) first, confirm **LED5** (link) on, then run this cell.

In [ ]:
eth_lock = threading.Lock()
eth_traces = {}
eth_rx = 0
eth_stop = threading.Event()

def _eth_worker():
    global eth_rx
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)
    bound = None
    for ip in usb_eth_bind_ips():
        try:
            sock.bind((ip, UDP_PORT))
            bound = ip
            break
        except OSError as e:
            print(f"bind {ip} failed: {e}")
    if bound is None:
        raise RuntimeError("Could not bind UDP/9 on USB Ethernet NIC")
    print(f"eth RX bound {bound}:{UDP_PORT}")
    sock.settimeout(0.5)
    pending = {}
    try:
        while not eth_stop.is_set():
            try:
                data, _addr = sock.recvfrom(2048)
            except socket.timeout:
                continue
            if len(data) < HDR_LEN:
                continue
            magic, ser, cidx, ccnt, nsamp = struct.unpack_from(HDR_FMT, data, 0)
            if magic != MAGIC:
                continue
            payload = data[HDR_LEN:HDR_LEN + 2 * nsamp]
            if len(payload) != 2 * nsamp:
                continue
            samples = np.frombuffer(payload, dtype="<u2").copy()
            slot = pending.setdefault(ser, {"chunks": {}, "n": ccnt})
            slot["chunks"][cidx] = samples
            slot["n"] = ccnt
            if len(slot["chunks"]) == slot["n"]:
                trace = np.concatenate([slot["chunks"][i] for i in range(slot["n"])])
                with eth_lock:
                    eth_traces[ser] = trace
                    eth_rx = len(eth_traces)
                del pending[ser]
    finally:
        sock.close()

eth_stop.clear()
eth_thread = threading.Thread(target=_eth_worker, daemon=True)
eth_thread.start()
print("eth RX listening")

## Run N encrypts + session_end (D16 / REG_HS_RST)

In [ ]:
cipher = AES.new(bytes(KEY), AES.MODE_ECB)
expected = list(cipher.encrypt(bytes(PLAINTEXT)))
key_w = bytearray(KEY[::-1])
pt_w  = bytearray(PLAINTEXT[::-1])

with eth_lock:
    eth_traces.clear()
    eth_rx = 0

t0 = time.time()
for i in range(N_TRACES):
    target.fpga_write(target.REG_CRYPT_KEY, key_w)
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt_w)
    scope.arm()
    target.fpga_write(target.REG_CRYPT_GO, [1])
    deadline = time.time() + 30.0
    while time.time() < deadline:
        busy = target.fpga_read(target.REG_CRYPT_GO, 1)[0]
        if busy == 0:
            break
        time.sleep(0.001)
    else:
        # One recovery attempt: clear HS + VCU session via D16
        print(f"trace {i}: busy stuck, REG_HS_RST then retry")
        target.fpga_write(REG_HS_RST, [1])
        time.sleep(0.2)
        target.fpga_write(target.REG_CRYPT_GO, [1])
        deadline = time.time() + 15.0
        while time.time() < deadline:
            busy = target.fpga_read(target.REG_CRYPT_GO, 1)[0]
            if busy == 0:
                break
            time.sleep(0.001)
        else:
            raise TimeoutError(
                f"trace {i}: still busy (eth_rx={eth_rx}). "
                "Check PMOD wires, LED5 link, SMA 100 MHz, VCU bit."
            )
    scope.capture()
    ct = bytearray(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))[::-1]
    assert list(ct) == expected, f"ct mismatch i={i}\n got {list(ct)}\n exp {expected}"
    if (i + 1) % 5 == 0 or i == 0:
        print(f"encrypt {i+1}/{N_TRACES}  eth_rx={eth_rx}  busy=0x00")

print(f"encrypt loop done in {time.time()-t0:.2f}s  eth_rx={eth_rx}")

target.fpga_write(REG_HS_RST, [1])
print("REG_HS_RST written (D16 done pulse, HS idle)")

drain_deadline = time.time() + 60.0
while time.time() < drain_deadline:
    with eth_lock:
        n = eth_rx
    if n >= N_TRACES:
        break
    print(f"  draining eth_rx={n}/{N_TRACES}")
    time.sleep(0.25)
else:
    raise TimeoutError(f"eth drain timeout: eth_rx={eth_rx}/{N_TRACES}")

time.sleep(0.15)
print(f"session complete: eth_rx={eth_rx}  unique serials={len(eth_traces)}")

## Plot

In [ ]:
serials = sorted(eth_traces.keys())
print("serials:", serials[:10], ("..." if len(serials) > 10 else ""))

def to_signed(u16):
    return u16.astype(np.int32) - (1 << 15)

fig, axes = plt.subplots(min(4, max(1, len(serials))), 1, figsize=(12, 8), sharex=True)
if not hasattr(axes, "__len__"):
    axes = [axes]
for ax, ser in zip(axes, serials[:len(axes)]):
    w = to_signed(eth_traces[ser])
    ax.plot(w, linewidth=0.7)
    ax.axvline(80, color="red", linestyle="--", linewidth=1)
    ax.set_ylabel(f"s={ser}")
    ax.set_title(f"serial {ser}  std={w.std():.1f}  mean={w.mean():.1f}")
axes[-1].set_xlabel("Sample (trig @ 80)")
plt.tight_layout()
plt.show()

if len(serials) >= 1:
    stack = np.stack([to_signed(eth_traces[s]) for s in serials])
    plt.figure(figsize=(12, 4))
    plt.plot(stack.mean(axis=0), linewidth=0.9)
    plt.axvline(80, color="red", linestyle="--", linewidth=1, label="trig 80")
    plt.title(f"Mean of {len(serials)} eth traces")
    plt.xlabel("Sample")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Throughput: 10k eth traces

Requires CW305 programmed, VCU programmed (LED5 up), and the **eth RX** cell still running.
Skips scope arm/capture so the rate reflects encrypt + UDP drain only.
Applies backpressure if the 16-slot VCU queue gets ahead of Ethernet.

In [ ]:
N_TP = 10_000
QUEUE_SLOTS = 16
BACKPRESSURE = QUEUE_SLOTS - 4  # wait if eth_rx lags encrypts by this much

cipher = AES.new(bytes(KEY), AES.MODE_ECB)
expected = list(cipher.encrypt(bytes(PLAINTEXT)))
key_w = bytearray(KEY[::-1])
pt_w = bytearray(PLAINTEXT[::-1])

with eth_lock:
    eth_traces.clear()
    eth_rx = 0

print(f"Starting throughput run: {N_TP} traces")
t_encrypt0 = time.time()
ok = 0
stuck = 0

for i in range(N_TP):
    # Backpressure: keep VCU queue from filling (can_accept would stall load)
    while True:
        with eth_lock:
            nrx = eth_rx
        if (i - nrx) < BACKPRESSURE:
            break
        time.sleep(0.0005)

    target.fpga_write(target.REG_CRYPT_KEY, key_w)
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt_w)
    target.fpga_write(target.REG_CRYPT_GO, [1])

    deadline = time.time() + 5.0
    busy = 1
    while time.time() < deadline:
        busy = target.fpga_read(target.REG_CRYPT_GO, 1)[0]
        if busy == 0:
            break
        time.sleep(0.0002)
    if busy != 0:
        stuck += 1
        target.fpga_write(REG_HS_RST, [1])
        time.sleep(0.05)
        target.fpga_write(target.REG_CRYPT_GO, [1])
        deadline = time.time() + 5.0
        while time.time() < deadline:
            busy = target.fpga_read(target.REG_CRYPT_GO, 1)[0]
            if busy == 0:
                break
            time.sleep(0.0002)
        if busy != 0:
            raise TimeoutError(
                f"throughput stuck at i={i}, eth_rx={eth_rx}, stuck={stuck}"
            )

    # Spot-check ciphertext occasionally (not every trace — keeps rate honest)
    if (i & 0x3FF) == 0:
        ct = bytearray(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))[::-1]
        assert list(ct) == expected, f"ct mismatch at i={i}"

    ok += 1
    if (i + 1) % 500 == 0 or i == 0:
        elapsed = time.time() - t_encrypt0
        with eth_lock:
            nrx = eth_rx
        print(
            f"  encrypt {i+1}/{N_TP}  eth_rx={nrx}  "
            f"enc={ok/elapsed:.1f}/s  rx={nrx/elapsed:.1f}/s"
        )

t_encrypt1 = time.time()
encrypt_s = t_encrypt1 - t_encrypt0
print(f"Encrypt loop done: {ok}/{N_TP} in {encrypt_s:.2f}s ({ok/encrypt_s:.1f} encrypts/s)")

# Session end + drain remaining UDP
target.fpga_write(REG_HS_RST, [1])
t_drain0 = time.time()
drain_deadline = t_drain0 + 120.0
while time.time() < drain_deadline:
    with eth_lock:
        nrx = eth_rx
    if nrx >= ok:
        break
    time.sleep(0.01)
else:
    raise TimeoutError(f"eth drain timeout: eth_rx={eth_rx}/{ok}")
t_drain1 = time.time()

total_s = t_drain1 - t_encrypt0
drain_s = t_drain1 - t_drain0
with eth_lock:
    nrx = eth_rx
    nser = len(eth_traces)

print("======== THROUGHPUT ========")
print(f"encrypts_ok     : {ok}/{N_TP}  (busy recoveries={stuck})")
print(f"eth_rx / serials: {nrx} / {nser}")
print(f"encrypt wall    : {encrypt_s:.2f}s  -> {ok/encrypt_s:.2f} encrypts/s")
print(f"drain wall      : {drain_s:.2f}s")
print(f"total wall      : {total_s:.2f}s  -> {nrx/total_s:.2f} traces/s (encrypt+drain)")
print(f"bits (rough)    : {nrx * 1024 * 16 / total_s / 1e6:.2f} Mbit/s payload @ 1024 x int16")
print("============================")

## Disconnect

In [ ]:
eth_stop.set()
eth_thread.join(timeout=2.0)
scope.dis()
target.dis()
print("Done")